# Tối ưu hóa: giải ví dụ của slide bằng code (cvxpy)

Notebook này đi kèm site *Tối ưu hóa phi tuyến & lồi* (Phenikaa). Mở trên **Google Colab** bằng nút “Open in Colab”; Colab có sẵn Python nên chạy được cả `cvxpy` (điều mà trình duyệt thuần không làm được).

**Quy ước nhân tử:** `cvxpy` trả `constraint.dual_value`; với ràng buộc dạng `g <= 0` đó chính là λ ≥ 0 trong `L = f + Σ λ_i g_i + Σ μ_j h_j`.

In [1]:
try:
    import cvxpy as cp
except ImportError:
    %pip install -q cvxpy
    import cvxpy as cp
import numpy as np, sympy as sp, itertools
print('cvxpy', cp.__version__)

cvxpy 1.7.1


## 1. Bài mẫu trong file `tam thoi la vay.docx`

`min (x-3)² + (y-4)² - 2` với `x² + y² ≤ 36`, `x ≥ 0`, `y ≥ 0`. Hàm mục tiêu và miền đều lồi (DCP), không cần `qcp=True`.

In [2]:
x, y = cp.Variable(), cp.Variable()
objective = cp.Minimize((x - 3)**2 + (y - 4)**2 - 2)
constraints = [x**2 + y**2 <= 36, x >= 0, y >= 0]
problem = cp.Problem(objective, constraints)
print('DCP (lồi theo quy tắc cvxpy):', problem.is_dcp())
problem.solve()
print('f* =', round(problem.value, 6), '| x =', round(float(x.value), 6), '| y =', round(float(y.value), 6))
print('nhân tử:', [round(abs(float(c.dual_value)), 6) for c in constraints])

DCP (lồi theo quy tắc cvxpy): True
f* = -2.0 | x = 3.0 | y = 4.0
nhân tử: [0.0, 0.0, 0.0]


**Kiểm tra bằng tay (3 dòng):** cực tiểu không ràng buộc là (3,4), `3²+4² = 25 ≤ 36` nên khả thi, mọi ràng buộc lỏng ⇒ mọi nhân tử bằng 0 ⇒ nghiệm (3,4), f* = −2.

## 2. Ví dụ 1 của slide 2 (tr.55): bài toán lồi, nhân tử (λ, μ) = (0, 1)

In [3]:
x, y = cp.Variable(), cp.Variable()
g = x + y - 2 <= 0
h = x - y + 1 == 0
p = cp.Problem(cp.Minimize((x - 1)**2 + y - 2), [g, h]); p.solve()
print('x*, y* =', round(float(x.value), 4), round(float(y.value), 4), '| f* =', round(p.value, 4))
print('lambda =', round(float(g.dual_value), 4), '| mu =', round(float(h.dual_value), 4))

x*, y* = 0.5 1.5 | f* = -0.25
lambda = 0.0 | mu = 1.0


## 3. Ví dụ 3 (tr.60): bài toán KHÔNG lồi — liệt kê điểm KKT bằng sympy

Thay cho bảng dài viết tay: chia trường hợp *ràng buộc chặt / lỏng*, giải hệ dừng + bù, rồi lọc nghiệm khả thi có λ ≥ 0.

In [4]:
def kkt_enumerate(f, ineq, eq, vars_):
    lam = sp.symbols(f'l1:{len(ineq)+1}', real=True)
    mu = sp.symbols(f'm1:{len(eq)+1}', real=True)
    L = f + sum(l*g for l, g in zip(lam, ineq)) + sum(m*h for m, h in zip(mu, eq))
    out = []
    for active in itertools.product([False, True], repeat=len(ineq)):
        eqs = [sp.diff(L, v) for v in vars_] + list(eq)
        for a, l, g in zip(active, lam, ineq):
            eqs.append(g if a else l)          # chặt: g=0 ; lỏng: lambda=0
        for sol in sp.solve(eqs, list(vars_) + list(lam) + list(mu), dict=True):
            if not all(v.is_real for v in sol.values()): continue
            feasible = all(g.subs(sol) <= 1e-9 for g in ineq)
            ok = feasible and all(sol[l] >= -1e-9 for l in lam)
            out.append(({v: sol[v] for v in vars_}, {str(l): sol[l] for l in lam}, ok))
    return out

X, Y = sp.symbols('x y', real=True)
for pt, lam, ok in kkt_enumerate(X*Y, [X**2 + Y**2 - 2], [], (X, Y)):
    print(pt, lam, 'f =', pt[X]*pt[Y], '->', 'KKT hợp lệ' if ok else 'loại')

{x: 0, y: 0} {'l1': 0} f = 0 -> KKT hợp lệ
{x: -1, y: -1} {'l1': -1/2} f = 1 -> loại
{x: 1, y: 1} {'l1': -1/2} f = 1 -> loại
{x: -1, y: 1} {'l1': 1/2} f = -1 -> KKT hợp lệ
{x: 1, y: -1} {'l1': 1/2} f = -1 -> KKT hợp lệ


## 4. Ví dụ 2 (tr.57): kiểm chứng điểm (1/3, 8/3)

Bài `min −(x²+y²+4x−6y)` với `x+y ≤ 3`, `−2x+y ≤ 2`.

In [5]:
F = -(X**2 + Y**2 + 4*X - 6*Y)
for pt, lam, ok in kkt_enumerate(F, [X + Y - 3, -2*X + Y - 2], [], (X, Y)):
    print(pt, lam, 'HỢP LỆ' if ok else 'loại')
print('Điểm (1/3, 8/3) có lambda_2 = -16/9 < 0 nên KHÔNG phải điểm KKT.')

{x: -2, y: 3} {'l1': 0, 'l2': 0} loại
{x: 0, y: 2} {'l1': 0, 'l2': -2} loại
{x: -1, y: 4} {'l1': 2, 'l2': 0} loại
{x: 1/3, y: 8/3} {'l1': 10/9, 'l2': -16/9} loại
Điểm (1/3, 8/3) có lambda_2 = -16/9 < 0 nên KHÔNG phải điểm KKT.


## 5. Ví dụ 4 (tr.61): quy hoạch tuyến tính

In [6]:
x, y = cp.Variable(), cp.Variable()
cons = [3*x + y <= 6, x + y <= 4, x >= 0, y >= 0]
for sense in (cp.Minimize, cp.Maximize):
    p = cp.Problem(sense(2*x + y), cons); p.solve()
    print(sense.__name__, '->', (round(float(x.value), 4), round(float(y.value), 4)), 'f* =', round(p.value, 4))

Minimize -> (0.0, 0.0) f* = 0.0
Maximize -> (1.0, 3.0) f* = 5.0


## 6. Tự làm

Đổi bài toán ở mục 1 thành `x² + y² ≤ 4` (miền nhỏ hơn). Nghiệm có còn là (3,4) không? Ràng buộc nào chặt, nhân tử bằng bao nhiêu? Dự đoán trước bằng tay (chiếu điểm (3,4) lên hình tròn bán kính 2), rồi kiểm bằng code.

In [7]:
# TODO: viết code của bạn ở đây
